# 07 — Profile antibiotic vocabulary and define reporting pairs

I profiled the antibiotics recorded for the reviewed culture organisms before selecting organism–drug pairs for the antibiogram. A drug appearing frequently in AST data does not automatically make it appropriate for a clinical-style summary.

The first three queries use source culture-organism records and reconciled endpoints. They are **not** patient-deduplicated antibiogram denominators.

### Antibiotics observed across reviewed source records

In [ ]:
%sql
select
    r.antibiotic,
    count(*) as endpoint_groups,
    count(distinct m.culture_organism_key) as culture_organisms,
    count(distinct m.patient_key) as patients

from workspace.micro_dev.int_mapped_cohort_culture_organism m

inner join workspace.micro_dev.int_susceptibility_reconciliation r
    on m.culture_organism_key = r.culture_organism_key

where m.include_primary_summary = 1
  and m.mapping_status = 'reviewed'
  and r.antibiotic is not null

group by r.antibiotic

order by endpoint_groups desc, r.antibiotic;

### Coverage of antibiotic results in the blood cohort

For each reporting organism, I compare the count of source culture-organism records with endpoints having a categorized S/R/I interpretation. Missing, conflicting, or noncategorical results do not count as categorized coverage.

In [ ]:
%sql
with eligible as (

    select
        cohort_id,
        organism_reporting_group,
        count(distinct culture_organism_key) as eligible_source_records

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where include_primary_summary = 1
      and mapping_status = 'reviewed'

    group by
        cohort_id,
        organism_reporting_group

),

endpoints as (

    select
        m.cohort_id,
        m.organism_reporting_group,
        r.antibiotic,

        count(*) as endpoint_groups,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as categorized_count,

        sum(
            case
                when r.resolution_status = 'conflict'
                    then 1
                else 0
            end
        ) as conflict_result_count,

        sum(
            case
                when r.resolution_status = 'missing'
                    then 1
                else 0
            end
        ) as missing_result_count

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on m.culture_organism_key = r.culture_organism_key

    where m.include_primary_summary = 1
      and m.mapping_status = 'reviewed'
      and r.antibiotic is not null

    group by
        m.cohort_id,
        m.organism_reporting_group,
        r.antibiotic

)

select
    e.organism_reporting_group,
    p.antibiotic,
    e.eligible_source_records,
    p.endpoint_groups,
    e.eligible_source_records - p.endpoint_groups as untested_source_records,
    p.categorized_count,
    p.conflict_result_count,
    p.missing_result_count,

    round(
        100.0 * p.categorized_count / e.eligible_source_records,
        1
    ) as categorized_result_coverage_pct

from endpoints p

inner join eligible e
    on p.cohort_id = e.cohort_id
   and p.organism_reporting_group = e.organism_reporting_group

where p.cohort_id = 'blood'

order by
    e.organism_reporting_group,
    p.categorized_count desc,
    p.antibiotic;

### Candidate drugs ranked by coverage

This ranking helped identify commonly tested agents. Final inclusion is a reviewed reporting choice, not an automated top-ten selection.

In [ ]:
%sql
with eligible as (

    select
        organism_reporting_group,
        count(distinct culture_organism_key) as eligible_source_records

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where cohort_id = 'blood'
      and include_primary_summary = 1
      and mapping_status = 'reviewed'

    group by organism_reporting_group

),

pair_counts as (

    select
        m.organism_reporting_group,
        r.antibiotic,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as categorized_count

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on m.culture_organism_key = r.culture_organism_key

    where m.cohort_id = 'blood'
      and m.include_primary_summary = 1
      and m.mapping_status = 'reviewed'
      and r.antibiotic is not null

    group by
        m.organism_reporting_group,
        r.antibiotic

),

ranked as (

    select
        p.organism_reporting_group,
        p.antibiotic,
        e.eligible_source_records,
        p.categorized_count,

        round(
            100.0 * p.categorized_count / e.eligible_source_records,
            1
        ) as categorized_result_coverage_pct,

        row_number() over (
            partition by p.organism_reporting_group
            order by
                p.categorized_count desc,
                p.antibiotic
        ) as antibiotic_rank

    from pair_counts p

    inner join eligible e
        on p.organism_reporting_group = e.organism_reporting_group

)

select *
from ranked
where antibiotic_rank <= 10

order by
    organism_reporting_group,
    antibiotic_rank;

### Check for simple label variants

I first look for antibiotic names that differ only in case or surrounding whitespace.

In [ ]:
%sql
select
    lower(trim(antibiotic)) as normalized_label,
    count(distinct antibiotic) as source_label_count,
    array_sort(collect_set(antibiotic)) as source_labels

from workspace.micro_dev.int_susceptibility_reconciliation

where antibiotic is not null

group by lower(trim(antibiotic))

having count(distinct antibiotic) > 1

order by source_label_count desc, normalized_label;

### Inspect the complete antibiotic vocabulary

I also generate a punctuation-insensitive comparison key for review. The normalized key is diagnostic; it does not replace the source label.

In [ ]:
%sql
with vocabulary as (

    select
        antibiotic,

        lower(
            trim(antibiotic)
        ) as case_space_normalized_label,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    antibiotic,
    case_space_normalized_label,
    punctuation_insensitive_key,
    endpoint_groups,
    culture_organisms

from vocabulary

order by antibiotic

### Identify possible duplicate names

A collision in the normalized keys would need a manual review before combining results from different source labels.

In [ ]:
%sql
with vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    punctuation_insensitive_key,

    count(
        distinct antibiotic
    ) as source_label_count,

    array_sort(
        collect_set(antibiotic)
    ) as source_labels,

    sum(endpoint_groups)
        as endpoint_groups

from vocabulary

group by punctuation_insensitive_key

having count(distinct antibiotic) > 1

order by
    source_label_count desc,
    punctuation_insensitive_key

### Review cohort-specific organism–antibiotic approvals

`reportable_organism_antibiotic_pairs` is the version-controlled, **cohort-specific**
allow-list. Only `reviewed` pairs with `include_primary_dashboard = 1` are
eligible for the reporting mart. The `organism_antibiotic_exclusions` seed
overrides that allow-list for defined organism–drug combinations.

These overrides affect the reporting panel, not the source AST facts or
the independent resistance-phenotype calculations.


In [ ]:
%sql
select
    p.organism_reporting_group,
    p.cohort_id,
    p.antibiotic,
    p.pair_order,
    case
        when e.organism_reporting_group is null then 'included'
        else 'suppressed_by_policy'
    end as reporting_decision,
    e.exclusion_reason,
    e.review_basis
from workspace.micro_dev.reportable_organism_antibiotic_pairs p
left join workspace.micro_dev.organism_antibiotic_exclusions e
    on p.organism_reporting_group = e.organism_reporting_group
   and p.antibiotic = e.antibiotic
where p.include_primary_dashboard = 1
  and p.review_status = 'reviewed'
order by
    p.organism_reporting_group,
    p.cohort_id,
    p.pair_order;


### Check the exclusion reference seed

The exclusion list is checked independently before the mart uses it. Every organism–drug pair must be unique and the required fields populated.

In [ ]:
%sql
select
    count(*) as row_count,

    count(distinct concat_ws(
        '||',
        organism_reporting_group,
        antibiotic
    )) as unique_pair_count,

    sum(
        case
            when organism_reporting_group is null
              or antibiotic is null
              or exclusion_reason is null
              or review_basis is null
                then 1
            else 0
        end
    ) as null_required_fields

from workspace.micro_dev.organism_antibiotic_exclusions;

The validated reporting seeds contain **33 distinct exclusion rules** and
**311 active cohort-specific approved organism–drug combinations**. Six of
those 311 combinations are intentionally suppressed for *Enterobacter
cloacae* complex under the project's AmpC display policy. The counts above
should be checked again after changes to the seeds.

### Confirm the phenotype-defining drug vocabulary

This diagnostic query inspects labels needed for CRE, ESCE-R, oxacillin
resistance, VRE, and MDR *P. aeruginosa*. It checks vocabulary availability,
**not** whether an organism–drug combination belongs in the reporting mart.
A missing drug is unavailable evidence, not automatically a negative test.


In [ ]:
%sql
with expected_antibiotics as (

    select *
    from values

        -- CRE phenotype
        ('CRE', 'Ertapenem'),
        ('CRE', 'Imipenem'),
        ('CRE', 'Meropenem'),
        ('CRE', 'Meropenem/Vaborbactam'),
        ('CRE', 'Imipenem/Relebactam'),

        -- Oxacillin-resistant Staphylococcus aureus
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Oxacillin'),
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Cefoxitin'),

        -- Vancomycin-resistant Enterococcus
        ('VRE', 'Vancomycin'),

        -- MDR Pseudomonas aeruginosa:
        -- extended-spectrum cephalosporins
        ('MDR_P_AERUGINOSA', 'Cefepime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime/Avibactam'),
        ('MDR_P_AERUGINOSA', 'Ceftolozane/Tazobactam'),

        -- fluoroquinolones
        ('MDR_P_AERUGINOSA', 'Ciprofloxacin'),
        ('MDR_P_AERUGINOSA', 'Levofloxacin'),

        -- aminoglycosides
        ('MDR_P_AERUGINOSA', 'Tobramycin'),

        -- carbapenems
        ('MDR_P_AERUGINOSA', 'Imipenem'),
        ('MDR_P_AERUGINOSA', 'Meropenem'),
        ('MDR_P_AERUGINOSA', 'Imipenem/Relebactam'),

        -- antipseudomonal penicillin combination
        ('MDR_P_AERUGINOSA', 'Piperacillin/Tazobactam'),

        -- siderophore cephalosporin
        ('MDR_P_AERUGINOSA', 'Cefiderocol'),
        
        -- NHSN extended-spectrum cephalosporin-resistant
        -- Enterobacterales phenotype
        ('ESCE_ENTEROBACTERALES', 'Cefepime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftriaxone'),
        ('ESCE_ENTEROBACTERALES', 'Cefotaxime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftazidime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftazidime/Avibactam'),
        ('ESCE_ENTEROBACTERALES', 'Ceftolozane/Tazobactam')
        

    as expected(
        phenotype_rule,
        expected_antibiotic
    )

),

expected_normalized as (

    select
        phenotype_rule,
        expected_antibiotic,

        regexp_replace(
            lower(trim(expected_antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key

    from expected_antibiotics

),

source_vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    e.phenotype_rule,
    e.expected_antibiotic,

    count(
        distinct v.antibiotic
    ) as matched_source_label_count,

    array_sort(
        collect_set(v.antibiotic)
    ) as matched_source_labels,

    coalesce(
        sum(v.endpoint_groups),
        0
    ) as endpoint_groups,

    coalesce(
        sum(v.culture_organisms),
        0
    ) as culture_organisms,

    case
        when count(distinct v.antibiotic) = 0
            then 'not_present'

        when count(distinct v.antibiotic) = 1
            then 'single_source_label'

        else 'multiple_candidate_labels'
    end as vocabulary_status

from expected_normalized e

left join source_vocabulary v
    on e.comparison_key = v.comparison_key

group by
    e.phenotype_rule,
    e.expected_antibiotic

order by
    e.phenotype_rule,
    e.expected_antibiotic

### Reassess the inactive organism–drug proposals using the overall population

There are **11 proposed combinations** with `include_primary_dashboard = 0`.
For each proposal, this query measures S/I/R coverage on the **earliest
overall culture per patient, cohort and reporting organism**. It is a
review aid, not an auto-approval algorithm. Each proposal appears once;
the retired phenotype-first reporting models are not required.

Review of interpretive guidance, selective testing, and source data is
still necessary before a proposal becomes approved.


In [ ]:
%sql
with proposed_pairs as (
    select
        p.organism_reporting_group,
        p.cohort_id,
        p.antibiotic,
        p.pair_order
    from workspace.micro_dev.reportable_organism_antibiotic_pairs p
    left join workspace.micro_dev.organism_antibiotic_exclusions e
        on p.organism_reporting_group = e.organism_reporting_group
       and p.antibiotic = e.antibiotic
    where p.review_status = 'proposed'
      and p.include_primary_dashboard = 0
      and e.organism_reporting_group is null
),
counts as (
    select
        p.organism_reporting_group,
        p.cohort_id,
        p.antibiotic,
        p.pair_order,
        count(f.overall_reporting_isolate_key) as eligible_count,
        sum(case
            when r.resolution_status = 'resolved'
             and r.susceptibility in (
                'Susceptible', 'Intermediate', 'Resistant'
             ) then 1 else 0
        end) as categorized_count,
        sum(case
            when r.resolution_status = 'conflict' then 1 else 0
        end) as conflict_count,
        sum(case
            when r.resolution_status = 'missing' then 1 else 0
        end) as missing_count
    from proposed_pairs p
    left join workspace.micro_dev.int_overall_first_reporting_isolate f
        on p.organism_reporting_group = f.organism_reporting_group
       and p.cohort_id = f.cohort_id
    left join workspace.micro_dev.int_overall_first_reporting_susceptibility r
        on f.overall_reporting_isolate_key = r.overall_reporting_isolate_key
       and p.antibiotic = r.antibiotic
    group by
        p.organism_reporting_group,
        p.cohort_id,
        p.antibiotic,
        p.pair_order
)
select
    organism_reporting_group,
    cohort_id,
    antibiotic,
    eligible_count,
    categorized_count,
    round(
        100.0 * categorized_count / nullif(eligible_count, 0),
        1
    ) as categorized_coverage_pct,
    conflict_count,
    missing_count,
    case
        when categorized_count >= 30 then 'meets_count_threshold'
        else 'insufficient_data'
    end as count_threshold_status
from counts
order by
    organism_reporting_group,
    cohort_id,
    pair_order;


### Decision and next step

The mart uses approved organism–cohort–drug rows after the separate
suppression override. This notebook shows **source vocabulary**, **coverage
diagnostics**, and **reference-table decisions**, but it does not edit a
seed or change the model. The 11 held combinations remain inactive unless
their reviewed CSV rows are deliberately revised and the dbt tests rerun.

No exploratory CSV is written by this notebook. Keep transient query
exports out of the tracked repository unless explicitly needed as a
documented analysis artifact.
